# DocuMentor 엔진 개발 - 개선된 코드

## 개선 내용
베이스라인 코드에서 BM25와 리랭킹 알고리즘이 적용

## 1. 환경 설정 및 라이브러리 설치

In [13]:
# 필요한 라이브러리 설치
!pip install -q pypdf2 langchain langchain-openai langchain-community faiss-cpu python-dotenv rank-bm25 sentence-transformers

In [14]:
# 라이브러리 임포트
import os
import json
from pathlib import Path
from typing import List, Dict, Tuple
import numpy as np

# PDF 파싱
from PyPDF2 import PdfReader

# 텍스트 분할
from langchain.text_splitter import RecursiveCharacterTextSplitter

# 임베딩 및 LLM
from langchain_openai import OpenAIEmbeddings, ChatOpenAI

# 벡터 스토어
import faiss

# BM25 검색
from rank_bm25 import BM25Okapi

# 리랭킹
from sentence_transformers import CrossEncoder

print("라이브러리 임포트 완료")

라이브러리 임포트 완료


In [15]:
# OpenAI API 키 설정
# 구글 코랩: 왼쪽 사이드바의 '🔑' 아이콘 클릭하여 시크릿에 추가
# 로컬: .env 파일에 OPENAI_API_KEY=your-api-key 형태로 저장

try:
    # 구글 코랩 환경
    from google.colab import userdata
    os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')
    print("API 키 로드 완료 (Colab)")
except:
    # 로컬 환경
    from dotenv import load_dotenv
    load_dotenv()
    print("API 키 로드 완료 (Local)")

# temp 폴더 생성
TEMP_DIR = Path("./temp")
TEMP_DIR.mkdir(exist_ok=True)
print(f"작업 폴더 생성: {TEMP_DIR}")

API 키 로드 완료 (Colab)
작업 폴더 생성: temp


## 2. 데이터 로드

구글 코랩의 경우 왼쪽 사이드바에서 PDF 파일을 업로드하거나,  
샘플 PDF를 다운로드하여 사용할 수 있습니다.

In [16]:
# 구글 코랩에서 파일 업로드
try:
    from google.colab import files
    uploaded = files.upload()
    pdf_path = list(uploaded.keys())[0]
    print(f"업로드 완료: {pdf_path}")
except:
    # 로컬 환경: 파일 경로 직접 지정
    pdf_path = "your_document.pdf"
    print(f"파일 경로: {pdf_path}")

Saving Skywork.pdf to Skywork (1).pdf
업로드 완료: Skywork (1).pdf


## 3. 파싱 (Parsing)

**Input**: PDF 파일 경로  
**Output**: 추출된 텍스트 문자열  
**저장**: `temp/parsed_text.txt`

In [17]:
def parse_pdf(pdf_path: str) -> str:
    """PDF에서 텍스트 추출"""
    reader = PdfReader(pdf_path)
    text = ""

    for page_num, page in enumerate(reader.pages, 1):
        page_text = page.extract_text()
        text += f"\n--- Page {page_num} ---\n{page_text}"

    return text

# PDF 파싱
print("PDF 파싱 중...")
parsed_text = parse_pdf(pdf_path)
print(f"파싱 완료: {len(parsed_text)} 글자")

# 저장
parsed_text_path = TEMP_DIR / "parsed_text.txt"
with open(parsed_text_path, 'w', encoding='utf-8') as f:
    f.write(parsed_text)
print(f"저장: {parsed_text_path}")

# 샘플 출력
print("\n[텍스트 샘플]")
print(parsed_text[:300] + "...")

PDF 파싱 중...
파싱 완료: 50200 글자
저장: temp/parsed_text.txt

[텍스트 샘플]

--- Page 1 ---
arXiv:2504.16656v4  [cs.CV]  6 Jun 2025Skywork R1V2: Multimodal Hybrid Reinforcement
Learning for Reasoning
Chris∗,Yichen Wei∗,Yi Peng ,Xiaokun Wang ,Weijie Qiu ,Wei Shen ,
Tianyidan Xie, Jiangbo Pei, Jianhao Zhang, Yunzhuo Hao, Xuchen Song†,
Yang Liu†, Yahui Zhou
Skywork AI, Kunlun ...


## 4. 청킹 (Chunking)

**Input**: 파싱된 텍스트  
**Output**: 텍스트 청크 리스트  
**저장**: `temp/chunks.json`

In [18]:
def chunk_text(text: str, chunk_size: int = 1000, chunk_overlap: int = 200) -> List[str]:
    """텍스트를 청크로 분할"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )

    chunks = text_splitter.split_text(text)
    return chunks

# 텍스트 로드
with open(parsed_text_path, 'r', encoding='utf-8') as f:
    parsed_text = f.read()

# 청킹
print("텍스트 청킹 중...")
chunks = chunk_text(parsed_text, chunk_size=1000, chunk_overlap=200)
print(f"청킹 완료: {len(chunks)}개 청크")

# 저장
chunks_path = TEMP_DIR / "chunks.json"
with open(chunks_path, 'w', encoding='utf-8') as f:
    json.dump({"chunks": chunks}, f, ensure_ascii=False, indent=2)
print(f"저장: {chunks_path}")

# 통계
chunk_lengths = [len(chunk) for chunk in chunks]
print(f"\n[청크 통계]")
print(f"평균 길이: {np.mean(chunk_lengths):.1f} 글자")
print(f"최소 길이: {np.min(chunk_lengths)} 글자")
print(f"최대 길이: {np.max(chunk_lengths)} 글자")

텍스트 청킹 중...
청킹 완료: 64개 청크
저장: temp/chunks.json

[청크 통계]
평균 길이: 951.6 글자
최소 길이: 508 글자
최대 길이: 998 글자


## 5. 임베딩

**Input**: 텍스트 청크 리스트  
**Output**: 임베딩 벡터 배열  
**저장**: `temp/embeddings.npy`

In [19]:
def generate_embeddings(chunks: List[str], batch_size: int = 100) -> np.ndarray:
    """텍스트 청크들의 임베딩 생성"""
    embeddings_model = OpenAIEmbeddings(model="text-embedding-3-large")

    embeddings = []
    total = len(chunks)

    print(f"임베딩 생성 중... (총 {total}개 청크)")
    for i in range(0, total, batch_size):
        batch = chunks[i:i+batch_size]
        embeddings.extend(embeddings_model.embed_documents(batch))
        print(f"진행: {min(i+batch_size, total)}/{total}", end='\r')

    print(f"\n임베딩 생성 완료: {total}개 청크")
    return np.array(embeddings, dtype=np.float32)

# 청크 로드
with open(chunks_path, 'r', encoding='utf-8') as f:
    chunks = json.load(f)["chunks"]

# 임베딩 생성
embeddings = generate_embeddings(chunks, batch_size=100)
print(f"Shape: {embeddings.shape}")
print(f"Dimension: {embeddings.shape[1]}차원")

# 저장
embeddings_path = TEMP_DIR / "embeddings.npy"
np.save(embeddings_path, embeddings)
print(f"저장: {embeddings_path}")

임베딩 생성 중... (총 64개 청크)
진행: 64/64
임베딩 생성 완료: 64개 청크
Shape: (64, 3072)
Dimension: 3072차원
저장: temp/embeddings.npy


## 6. 벡터 DB 구축 (FAISS)

**Input**: 임베딩 벡터 배열  
**Output**: FAISS 인덱스  
**저장**: `temp/faiss_index.bin`

In [20]:
def build_faiss_index(embeddings: np.ndarray) -> faiss.Index:
    """FAISS 인덱스 구축"""
    dimension = embeddings.shape[1]

    # L2 거리 기반 인덱스 생성
    index = faiss.IndexFlatL2(dimension)

    # 정규화 (코사인 유사도)
    faiss.normalize_L2(embeddings)

    # 벡터 추가
    index.add(embeddings)

    return index

# 임베딩 로드
embeddings = np.load(embeddings_path)

# 인덱스 구축
print("FAISS 인덱스 구축 중...")
faiss_index = build_faiss_index(embeddings)
print(f"인덱스 구축 완료")
print(f"총 벡터 수: {faiss_index.ntotal}")
print(f"차원: {faiss_index.d}")

# 저장
faiss_index_path = TEMP_DIR / "faiss_index.bin"
faiss.write_index(faiss_index, str(faiss_index_path))
print(f"저장: {faiss_index_path}")

FAISS 인덱스 구축 중...
인덱스 구축 완료
총 벡터 수: 64
차원: 3072
저장: temp/faiss_index.bin


In [21]:
def build_bm25_index(chunks: List[str]) -> BM25Okapi:
    """
    BM25 인덱스 구축 (키워드 기반 검색)

    Args:
        chunks: 텍스트 청크 리스트

    Returns:
        BM25Okapi 인덱스
    """
    # 각 청크를 토큰화 (공백 기준)
    tokenized_chunks = [chunk.split() for chunk in chunks]

    # BM25 인덱스 생성
    bm25_index = BM25Okapi(tokenized_chunks)

    return bm25_index

# 청크 로드
with open(chunks_path, 'r', encoding='utf-8') as f:
    chunks = json.load(f)["chunks"]

# BM25 인덱스 구축
print("BM25 인덱스 구축 중...")
bm25_index = build_bm25_index(chunks)
print(f"BM25 인덱스 구축 완료")

# BM25 인덱스 저장
import pickle
bm25_index_path = TEMP_DIR / "bm25_index.pkl"
with open(bm25_index_path, 'wb') as f:
    pickle.dump(bm25_index, f)
print(f"저장: {bm25_index_path}")

BM25 인덱스 구축 중...
BM25 인덱스 구축 완료
저장: temp/bm25_index.pkl


## 7. 검색 시스템

**Input**: 사용자 쿼리  
**Output**: 관련성 높은 청크들 (top-k)

In [22]:
class HybridRetriever:
    """FAISS(벡터) + BM25(키워드) + 리랭킹 하이브리드 검색"""

    def __init__(self, faiss_index, bm25_index, chunks, embeddings_model):
        self.faiss_index = faiss_index
        self.bm25_index = bm25_index
        self.chunks = chunks
        self.embeddings_model = embeddings_model

        # 리랭킹 모델 초기화
        print("리랭킹 모델 로딩 중...")
        self.reranker = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')
        print("리랭킹 모델 준비 완료")

    def search_faiss(self, query: str, top_k: int = 10) -> List[Tuple[int, float]]:
        """FAISS 벡터 검색"""
        # 쿼리 임베딩
        query_embedding = self.embeddings_model.embed_query(query)
        query_vector = np.array([query_embedding], dtype=np.float32)

        # 정규화
        faiss.normalize_L2(query_vector)

        # 검색
        distances, indices = self.faiss_index.search(query_vector, top_k)

        # 유사도 변환
        similarities = 1 - (distances[0] ** 2) / 2

        return [(int(idx), float(sim)) for idx, sim in zip(indices[0], similarities)]

    def search_bm25(self, query: str, top_k: int = 10) -> List[Tuple[int, float]]:
        """BM25 키워드 검색"""
        # 쿼리 토큰화
        tokenized_query = query.split()

        # BM25 점수 계산
        bm25_scores = self.bm25_index.get_scores(tokenized_query)

        # 상위 k개 추출
        top_indices = np.argsort(bm25_scores)[-top_k:][::-1]

        # 점수 정규화 (0~1)
        max_score = bm25_scores.max() if bm25_scores.max() > 0 else 1
        normalized_scores = bm25_scores / max_score

        return [(int(idx), float(normalized_scores[idx])) for idx in top_indices]

    def hybrid_search(self, query: str, top_k: int = 20,
                     alpha: float = 0.5) -> List[Tuple[int, float]]:
        """
        하이브리드 검색 (FAISS + BM25)

        Args:
            query: 검색 쿼리
            top_k: 각 검색에서 가져올 개수
            alpha: FAISS 가중치 (0~1, BM25 가중치는 1-alpha)

        Returns:
            [(청크_인덱스, 최종_점수), ...]
        """
        # 1. FAISS 검색
        faiss_results = self.search_faiss(query, top_k)

        # 2. BM25 검색
        bm25_results = self.search_bm25(query, top_k)

        # 3. 점수 결합
        combined_scores = {}

        # FAISS 점수 반영
        for idx, score in faiss_results:
            combined_scores[idx] = alpha * score

        # BM25 점수 반영
        for idx, score in bm25_results:
            if idx in combined_scores:
                combined_scores[idx] += (1 - alpha) * score
            else:
                combined_scores[idx] = (1 - alpha) * score

        # 점수 기준 정렬
        sorted_results = sorted(combined_scores.items(),
                               key=lambda x: x[1],
                               reverse=True)

        return sorted_results[:top_k]

    def rerank(self, query: str, candidates: List[Tuple[int, float]],
               top_k: int = 3) -> List[Dict]:
        """
        리랭킹으로 최종 결과 정제 (순위만 사용, 점수 제외)

        Returns:
            [{'chunk_id': 0, 'rank': 1, 'text': '...'}, ...]
        """
        # 쿼리-청크 쌍 생성
        pairs = [(query, self.chunks[idx]) for idx, _ in candidates]

        # 리랭킹 점수 계산
        rerank_scores = self.reranker.predict(pairs)

        # 점수와 인덱스 결합
        reranked_results = [
            (idx, float(score))
            for (idx, _), score in zip(candidates, rerank_scores)
        ]

        # 점수 기준 정렬 (점수는 내부적으로만 사용)
        reranked_results.sort(key=lambda x: x[1], reverse=True)

        # 상위 k개 반환 (순위만 표시)
        final_results = []
        for rank, (idx, _) in enumerate(reranked_results[:top_k], 1):
            final_results.append({
                'chunk_id': idx,
                'rank': rank,
                'text': self.chunks[idx]
            })

        return final_results

    def search(self, query: str, top_k: int = 3,
               hybrid_top_k: int = 20, alpha: float = 0.5) -> List[Dict]:
        """
        전체 검색 파이프라인: 하이브리드 검색 + 리랭킹

        Args:
            query: 검색 쿼리
            top_k: 최종 반환할 개수
            hybrid_top_k: 하이브리드 검색에서 가져올 후보 개수
            alpha: FAISS 가중치
        """
        # 1. 하이브리드 검색
        print(f"하이브리드 검색 중... (FAISS + BM25)")
        candidates = self.hybrid_search(query, top_k=hybrid_top_k, alpha=alpha)
        print(f"후보 추출: {len(candidates)}개")

        # 2. 리랭킹
        print(f"리랭킹 중...")
        final_results = self.rerank(query, candidates, top_k=top_k)
        print(f"최종 선택: {top_k}개")

        return final_results


# 데이터 로드
faiss_index = faiss.read_index(str(faiss_index_path))
with open(bm25_index_path, 'rb') as f:
    bm25_index = pickle.load(f)
with open(chunks_path, 'r', encoding='utf-8') as f:
    chunks = json.load(f)["chunks"]

# 임베딩 모델 초기화
embeddings_model = OpenAIEmbeddings(model="text-embedding-3-large")

# 하이브리드 검색기 초기화
print("\n하이브리드 검색 시스템 초기화 중...")
hybrid_retriever = HybridRetriever(
    faiss_index=faiss_index,
    bm25_index=bm25_index,
    chunks=chunks,
    embeddings_model=embeddings_model
)
print("검색 시스템 준비 완료")


하이브리드 검색 시스템 초기화 중...
리랭킹 모델 로딩 중...
리랭킹 모델 준비 완료
검색 시스템 준비 완료


## 8. 검색된 내용 기반의 LLM 응답 생성

**Input**: 사용자 질문  
**Output**: LLM이 생성한 답변

In [23]:
def generate_answer(query: str, top_k: int = 3,
                   hybrid_top_k: int = 20, alpha: float = 0.5) -> Dict[str, any]:
    """
    RAG 파이프라인: 하이브리드 검색 + 리랭킹 + LLM 답변

    Args:
        query: 질문
        top_k: 최종 반환할 청크 개수
        hybrid_top_k: 하이브리드 검색 후보 개수
        alpha: FAISS 가중치 (0=BM25만, 1=FAISS만, 0.5=균등)
    """
    # 1. 하이브리드 검색 + 리랭킹
    print(f"질문: {query}\n")
    retrieved_chunks = hybrid_retriever.search(
        query,
        top_k=top_k,
        hybrid_top_k=hybrid_top_k,
        alpha=alpha
    )

    # 검색 결과 출력
    print(f"\n검색 완료: {len(retrieved_chunks)}개 청크 선택\n")
    for chunk in retrieved_chunks:
        print(f"[{chunk['rank']}위] 청크 #{chunk['chunk_id']}")
        print(f"{chunk['text'][:150]}...\n")

    # 2. 컨텍스트 구성
    context = "\n\n".join([
        f"[문서 {chunk['rank']}]\n{chunk['text']}"
        for chunk in retrieved_chunks
    ])

    # 3. 프롬프트 생성
    prompt = f"""당신은 문서 기반 질의응답 시스템입니다. 주어진 문서 내용을 바탕으로 질문에 답변하세요.

규칙:
1. 반드시 제공된 문서 내용만을 사용하여 답변하세요.
2. 문서에 정보가 없으면 "제공된 문서에서 관련 정보를 찾을 수 없습니다"라고 답변하세요.
3. 답변은 명확하고 간결하게 작성하세요.

문서 내용:
{context}

질문: {query}

답변:"""

    # 4. LLM 호출
    print("답변 생성 중...")
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
    response = llm.invoke(prompt)
    answer = response.content

    # 5. 결과 반환
    result = {
        'query': query,
        'answer': answer,
        'retrieved_chunks': retrieved_chunks,
        'sources': [(chunk['rank'], chunk['chunk_id']) for chunk in retrieved_chunks]
    }

    return result

print("RAG 시스템 준비 완료")

RAG 시스템 준비 완료


## 9. 질의응답 테스트


In [24]:
# 질문 입력
query = "Table 1에서 LiveCode brnch에서 Proprietary Models 중 가장 높은 점수를 받은 모델을 알려줘."  # 원하는 질문으로 변경하세요

# RAG 실행
# alpha 조정: 0.5=균등, 0.7=벡터 중심, 0.3=키워드 중심
result = generate_answer(
    query,
    top_k=3,           # 최종 반환할 청크 개수
    hybrid_top_k=20,   # 하이브리드 검색 후보 개수
    alpha=0.5          # FAISS 가중치
)

# 결과 출력
print("\n" + "="*80)
print("[최종 답변]")
print("="*80)
print(result['answer'])

질문: Table 1에서 LiveCode brnch에서 Proprietary Models 중 가장 높은 점수를 받은 모델을 알려줘.

하이브리드 검색 중... (FAISS + BM25)
후보 추출: 20개
리랭킹 중...
최종 선택: 3개

검색 완료: 3개 청크 선택

[1위] 청크 #34
Model MMMU Math- Math- Olympiad AIME LiveCode Live IFEV AL
Vista Vision Bench 24 bench Bench
Proprietary Models
Claude-3.5-Sonnet 70.4 67.7 - - - - - ...

[2위] 청크 #29
any variations in model outputs.
Baselines We conduct comprehensive evaluations against several strong proprietary models, including
Claude-3.5-Sonnet...

[3위] 청크 #33
74.0% surpasses Claude 3.5 Sonnet (67.7%) and is competitive with Gemini 2 Flash (73.1%) and
Kimi k1.5 longcot (74.9%).
While larger proprietary model...

답변 생성 중...

[최종 답변]
Table 1에서 LiveCode Bench에서 Proprietary Models 중 가장 높은 점수를 받은 모델은 OpenAI-o4-mini로, 점수는 74.6입니다.


In [25]:
# 질문 입력
query = "Figure2에서 BFVL 벤치마크에서 R1V-1-38B 모델의 성능 점수를 각각 알려줘."  # 원하는 질문으로 변경하세요

# RAG 실행
# alpha 조정: 0.5=균등, 0.7=벡터 중심, 0.3=키워드 중심
result = generate_answer(
    query,
    top_k=3,           # 최종 반환할 청크 개수
    hybrid_top_k=20,   # 하이브리드 검색 후보 개수
    alpha=0.5          # FAISS 가중치
)

# 결과 출력
print("\n" + "="*80)
print("[최종 답변]")
print("="*80)
print(result['answer'])

질문: Figure2에서 BFVL 벤치마크에서 R1V-1-38B 모델의 성능 점수를 각각 알려줘.

하이브리드 검색 중... (FAISS + BM25)
후보 추출: 20개
리랭킹 중...
최종 선택: 3개

검색 완료: 3개 청크 선택

[1위] 청크 #30
other state-of-the-art models across various text reasoning benchmarks. Skywork R1V2 demonstrates
exceptional reasoning capabilities, achieving 78.9% ...

[2위] 청크 #31
(66.3% vs. 60.3%). This suggests that our approach enables efficient learning with fewer parameters,
making R1V2 a more practical choice for deploymen...

[3위] 청크 #40
determining AC frequency, showcasing its understanding of electromagnetic induction principles.
The model’s systematic elimination of incorrect option...

답변 생성 중...

[최종 답변]
제공된 문서에서 관련 정보를 찾을 수 없습니다.


## 10. 이미지 활용


In [26]:
import base64
from pathlib import Path
from typing import List

# 이미지 업로드
try:
    from google.colab import files
    print("이미지 파일을 업로드하세요 (여러 파일 선택 가능)")
    uploaded_images = files.upload()

    # 업로드된 이미지 경로 저장
    image_paths = []
    for filename in uploaded_images.keys():
        image_paths.append(filename)
        print(f"업로드 완료: {filename}")

    print(f"\n총 {len(image_paths)}개 이미지 업로드됨")

except:
    # 로컬 환경: 이미지 폴더 지정
    image_folder = "./images"  # 이미지가 있는 폴더 경로
    image_extensions = {'.jpg', '.jpeg', '.png', '.webp'}

    image_paths = []
    if Path(image_folder).exists():
        for img_path in Path(image_folder).iterdir():
            if img_path.suffix.lower() in image_extensions:
                image_paths.append(str(img_path))
        print(f"이미지 폴더에서 {len(image_paths)}개 이미지 발견")
    else:
        print(f"이미지 폴더가 없습니다: {image_folder}")
        image_paths = []

이미지 파일을 업로드하세요 (여러 파일 선택 가능)


Saving skywork_figure2.PNG to skywork_figure2.PNG
업로드 완료: skywork_figure2.PNG

총 1개 이미지 업로드됨


In [27]:
# ==========================================
# 멀티모달 RAG (텍스트 + 이미지)
# ==========================================

def encode_image(image_path: str) -> str:
    """이미지를 base64로 인코딩"""
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode('utf-8')


def generate_answer_with_images(
    query: str,
    image_paths: List[str] = None,
    top_k: int = 3,
    hybrid_top_k: int = 20,
    alpha: float = 0.5
) -> Dict[str, any]:
    """
    멀티모달 RAG: 텍스트 검색 + 이미지 분석

    Args:
        query: 사용자 질문
        image_paths: 이미지 파일 경로 리스트 (None이면 텍스트만)
        top_k: 검색할 텍스트 청크 개수
        hybrid_top_k: 하이브리드 검색 후보 개수
        alpha: FAISS 가중치
    """

    # 1. 텍스트 검색
    print(f"질문: {query}\n")
    print("텍스트 검색 중...")
    retrieved_chunks = hybrid_retriever.search(
        query,
        top_k=top_k,
        hybrid_top_k=hybrid_top_k,
        alpha=alpha
    )

    # 텍스트 컨텍스트 구성
    text_context = "\n\n".join([
        f"[텍스트 {chunk['rank']}]\n{chunk['text']}"
        for chunk in retrieved_chunks
    ])

    print(f"텍스트 검색 완료: {len(retrieved_chunks)}개 청크\n")

    # 2. 이미지가 있는 경우
    if image_paths and len(image_paths) > 0:
        print(f"이미지 처리 중: {len(image_paths)}개")

        image_contents = []
        valid_images = []

        for i, img_path in enumerate(image_paths, 1):
            if Path(img_path).exists():
                try:
                    base64_image = encode_image(img_path)
                    image_contents.append({
                        "type": "image_url",
                        "image_url": {
                            "url": f"data:image/jpeg;base64,{base64_image}"
                        }
                    })
                    valid_images.append(Path(img_path).name)
                    print(f"  - 이미지 {i}: {Path(img_path).name}")
                except Exception as e:
                    print(f"  - 이미지 {i}: 처리 실패 ({str(e)})")
            else:
                print(f"  - 이미지 {i}: 파일 없음")

        # 3. 멀티모달 프롬프트
        if image_contents:
            prompt_text = f"""당신은 문서와 이미지를 분석하는 질의응답 시스템입니다.

작업:
1. 제공된 텍스트 문서를 분석하세요.
2. 이미지들을 자세히 관찰하세요.
3. 텍스트와 이미지를 종합하여 질문에 답변하세요.
4. 이미지에서 발견한 내용을 구체적으로 언급하세요.

텍스트 문서:
{text_context}

질문: {query}

답변:"""

            print("\n답변 생성 중 (텍스트 + 이미지 분석)...")

            llm = ChatOpenAI(
                model="gpt-4o",  # GPT-4 Vision
                temperature=0,
                max_tokens=1000
            )

            # 메시지 구성
            messages = [
                {
                    "role": "user",
                    "content": [
                        {"type": "text", "text": prompt_text},
                        *image_contents
                    ]
                }
            ]

            response = llm.invoke(messages)
            answer = response.content

            result = {
                'query': query,
                'answer': answer,
                'text_sources': [(chunk['rank'], chunk['chunk_id']) for chunk in retrieved_chunks],
                'image_sources': valid_images,
                'mode': 'multimodal'
            }

            return result

    # 4. 이미지가 없는 경우 (텍스트만)
    prompt = f"""당신은 문서 기반 질의응답 시스템입니다.

규칙:
1. 제공된 문서 내용만 사용하여 답변하세요.
2. 정보가 없으면 명시하세요.

문서 내용:
{text_context}

질문: {query}

답변:"""

    print("\n답변 생성 중 (텍스트만)...")

    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
    response = llm.invoke(prompt)
    answer = response.content

    result = {
        'query': query,
        'answer': answer,
        'text_sources': [(chunk['rank'], chunk['chunk_id']) for chunk in retrieved_chunks],
        'image_sources': [],
        'mode': 'text_only'
    }

    return result

print("멀티모달 RAG 준비 완료")

멀티모달 RAG 준비 완료


In [29]:
# ==========================================
# 멀티모달 RAG 테스트
# ==========================================

# 질문 입력
query = "Figure2에서 BFVL 벤치마크에서 R1V-1-38B 모델의 성능 점수를 각각 알려줘."

# 멀티모달 RAG 실행
result = generate_answer_with_images(
    query=query,
    image_paths=image_paths,  # 앞서 업로드한 이미지
    top_k=3,
    alpha=0.5
)

# 결과 출력
print("\n" + "="*80)
print(f"[최종 답변 - {result['mode']}]")
print("="*80)
print(result['answer'])

질문: Figure2에서 BFVL 벤치마크에서 R1V-1-38B 모델의 성능 점수를 각각 알려줘.

텍스트 검색 중...
하이브리드 검색 중... (FAISS + BM25)
후보 추출: 20개
리랭킹 중...
최종 선택: 3개
텍스트 검색 완료: 3개 청크

이미지 처리 중: 1개
  - 이미지 1: skywork_figure2.PNG

답변 생성 중 (텍스트 + 이미지 분석)...

[최종 답변 - multimodal]
Figure 2에서 BFCL 벤치마크에서 R1V-1-38B 모델의 성능 점수는 53.5%입니다.
